# **Bluestone Real Estate**

## Identify Tables:

The dataset consists of four tables:
- Sales table (salestable.xlsx). This contains completed property sales
- Rental table (rentaltable.xlsx). This contains completed rental transactions and lease details
- listings table (listings.xlsx). This contains active and historical property listings.
- Inquiry table (inquiry.xlsx). this contains buyer and renter inquiries with lead scoring and conversion tracking.

## Goal:

The goal is to automate decision-making through predictive models and
data-driven analytics.

## Target:

- Develop, train, evaluate and deploy a machine learning model that predicts property prices, rental demand and provides insights based on marketing trends.

## Import Libraries:

In [1]:
# Core
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

# Load Data into Notebook

In [6]:
salestable = pd.read_excel('Data/salestable.xlsx')
rentaltable = pd.read_excel('Data/rentaltable.xlsx')
listingstable = pd.read_excel('Data/listingstable.xlsx')
inquirytable = pd.read_excel('Data/inquirytable.xlsx')

# Examining Salestable Dataset

In [7]:
salestable.head()

,CITY,STATE,ZIPCODE,BUYER_NAME,INQUIRY_ID,LIST_PRICE,LISTING_ID,OFFER_DATE,OFFER_PRICE,DAYS_TO_CLOSE,...,TRANSACTION_ID,FINAL_SALE_PRICE,MARKET_AVG_PRICE,EARNEST_MONEY_PCT,TRANSACTION_TYPE,OFFER_TO_LIST_RATIO,CONTINGENCY_APPRAISAL,CONTINGENCY_FINANCING,CONTINGENCY_INSPECTION,CLOSE_DATE
0,Austin,TX,78745,Jerry Thomas,5609618a-8289-43d1-aad7-11f003e053dc,499000,"4710-Roundup-Trl,-Austin,-TX-78745",2025-10-29 15:09:00,523700,33,...,7f4a2380-362d-4927-87a2-33d716b9e232,521400,538887,0.012,Sale,1.0495,False,True,False,2025-12-04
1,Austin,TX,78745,Denise Morrison,bf37c9ae-c798-4257-88b0-d852aa98e641,595000,"4709-Mount-Vernon-Dr,-Austin,-TX-78745",2026-03-02 13:22:00,563600,33,...,17ffa475-cea7-4a04-a25e-c524b25e76ee,551000,538887,0.025,Sale,0.9472,False,True,True,2026-04-09
2,Austin,TX,78745,Jennifer Clayton,3ffe157b-7d2d-4e56-b5b1-fac4ac4f9259,490000,"8121-Evadean-Cir,-Austin,-TX-78745",2026-03-20 13:01:00,489700,37,...,83984cc8-4f24-4e03-b6a2-7bb08f623e98,485300,538887,0.021,Sale,0.9994,True,True,True,2026-04-30
3,Austin,TX,78745,Susan Vasquez,4c0360f9-3ced-4beb-afd2-724afeded806,469995,"1444-Salem-Meadow-Cir,-Austin,-TX-78745",2026-03-27 15:09:00,466500,44,...,d80dbae8-3cb6-4986-a978-0cf6227aa592,459800,538887,0.013,Sale,0.9926,True,True,True,2026-05-11
4,Austin,TX,78745,April Edwards,d5b247a5-da2f-46ff-8a70-f23153ef83f3,514000,"2817-Norfolk-Dr,-Austin,-TX-78745",2026-03-26 13:10:00,492000,41,...,271cdb34-fe9c-470f-9ac9-552d14c17e40,480000,538887,0.017,Sale,0.9572,False,True,False,2026-05-07


In [8]:
salestable.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43304 entries, 0 to 43303
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   CITY                    43304 non-null  object        
 1   STATE                   43304 non-null  object        
 2   ZIPCODE                 43304 non-null  int64         
 3   BUYER_NAME              43304 non-null  object        
 4   INQUIRY_ID              43304 non-null  object        
 5   LIST_PRICE              43304 non-null  int64         
 6   LISTING_ID              43304 non-null  object        
 7   OFFER_DATE              43304 non-null  datetime64[ns]
 8   OFFER_PRICE             43304 non-null  int64         
 9   DAYS_TO_CLOSE           43304 non-null  int64         
 10  ACCEPTED_DATE           43304 non-null  datetime64[ns]
 11  MARKET_AVG_DOM          43304 non-null  float64       
 12  PROPERTY_TYPE           43304 non-null  object

In [9]:
salestable.describe()

,ZIPCODE,LIST_PRICE,OFFER_DATE,OFFER_PRICE,DAYS_TO_CLOSE,ACCEPTED_DATE,MARKET_AVG_DOM,FINAL_SALE_PRICE,MARKET_AVG_PRICE,EARNEST_MONEY_PCT,OFFER_TO_LIST_RATIO,CLOSE_DATE
count,43304.000000,4.330400e+04,43304,4.330400e+04,43304.000000,43304,43304.000000,4.330400e+04,4.330400e+04,43304.000000,43304.000000,43304
mean,64266.180422,1.185194e+06,2026-02-21 16:17:09.240254976,1.126233e+06,37.068400,2026-02-24 16:17:23.206632192,78.135408,1.109192e+06,6.076221e+05,0.019961,0.969040,2026-04-02 03:52:34.406059520
min,28202.000000,8.000000e+03,2017-12-31 12:40:00,8.000000e+03,7.000000,2018-01-01 12:40:00,26.680000,7.900000e+03,1.399000e+05,0.010000,0.910000,2018-02-08 00:00:00
25%,60610.000000,3.747578e+05,2026-02-01 10:26:30,3.659000e+05,33.000000,2026-02-04 10:56:00,68.600000,3.604000e+05,3.599690e+05,0.015000,0.934800,2026-03-11 00:00:00
50%,77008.000000,5.949140e+05,2026-03-09 14:46:00,5.704000e+05,39.000000,2026-03-12 13:59:00,78.130000,5.614000e+05,4.843440e+05,0.020000,0.958600,2026-04-18 00:00:00
75%,80209.000000,1.150000e+06,2026-04-09 17:35:45,1.095725e+06,44.000000,2026-04-12 16:08:30,87.780000,1.078700e+06,7.581220e+05,0.025000,1.002400,2026-05-20 00:00:00
max,86323.000000,3.500000e+07,2026-06-05 17:16:00,3.382810e+07,60.000000,2026-06-09 17:16:00,249.500000,3.341560e+07,2.896378e+06,0.030000,1.060200,2026-07-30 00:00:00
std,20005.550239,2.021730e+06,NaN,1.893623e+06,10.619914,NaN,16.699616,1.863848e+06,3.803082e+05,0.005772,0.041857,NaN


# Examining Rentaltable Dataset

In [10]:
rentaltable.head()

,CITY,STATE,ZIPCODE,INQUIRY_ID,LISTING_ID,AGREED_RENT,LISTED_RENT,TENANT_NAME,PETS_ALLOWED,LEASE_END_DATE,...,LEASE_START_DATE,LEASE_TERM(MNTS),RENT_TO_LIST_RATIO,TRANSACTION_TYPE,SCREENING_OUTCOME,SECURITY_DEPOSIT_AMT,SECURITY_DEPOSIT_MONTHS,SECURITY_DEPOSIT_MONTHS_ORIGINAL,APPROVED_DATE,APPLICATION_DATE
0,Chicago,IL,60613,a4d8c066-2e31-4dde-907d-12d1a99d3b3d,"4334-N-Hazel-St,-Apt-1213,-Chicago,-IL-60613",1650,1685,Brooke Phillips,False,2026-02-26,...,2025-04-02,11,0.9792,Rental,Approved,1650,1,1,2025-03-23,2025-03-21
1,Chicago,IL,60625,f7d0451b-2718-4958-92e9-023ce22c1f53,"3442-W-Foster-Ave,-Apt-1S,-Chicago,-IL-60625",1800,1850,Rebecca Hughes,False,2026-09-11,...,2025-10-16,11,0.9730,Rental,Approved,1800,1,1,2025-10-09,2025-10-07
2,Chicago,IL,60613,dce6f21b-5c5c-47d6-acaf-f3683fe946a0,"3710-N-Pine-Grove-Ave,-Apt-514,-Chicago,-IL-60613",1760,1800,Daniel Harris,False,2026-07-17,...,2025-08-21,11,0.9778,Rental,Approved,1760,1,1,2025-08-14,2025-08-09
3,Chicago,IL,60626,896ba5d2-e65d-4f5e-8d17-71d79cf2f244,"1329-W-Estes-Ave,-Apt-1A,-Chicago,-IL-60626",3050,3071,William Paul,False,2026-10-03,...,2025-11-07,11,0.9932,Rental,Approved,3050,1,1,2025-10-25,2025-10-21
4,Chicago,IL,60608,19e0b31c-e63a-4547-bf7d-9b5060377846,"1437-W-Cullerton-St,-Chicago,-IL-60608",2860,2895,Molly Glover,False,2027-01-30,...,2026-03-06,11,0.9879,Rental,Approved,2860,1,1,2026-03-03,2026-03-02


In [11]:
rentaltable.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7051 entries, 0 to 7050
Data columns (total 22 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   CITY                              7051 non-null   object        
 1   STATE                             7051 non-null   object        
 2   ZIPCODE                           7051 non-null   int64         
 3   INQUIRY_ID                        7051 non-null   object        
 4   LISTING_ID                        7051 non-null   object        
 5   AGREED_RENT                       7051 non-null   int64         
 6   LISTED_RENT                       7051 non-null   int64         
 7   TENANT_NAME                       7051 non-null   object        
 8   PETS_ALLOWED                      7051 non-null   bool          
 9   LEASE_END_DATE                    7051 non-null   datetime64[ns]
 10  PROPERTY_TYPE                     7051 non-null 

In [12]:
rentaltable.describe()

,ZIPCODE,AGREED_RENT,LISTED_RENT,LEASE_END_DATE,LEASE_START_DATE,LEASE_TERM(MNTS),RENT_TO_LIST_RATIO,SECURITY_DEPOSIT_AMT,SECURITY_DEPOSIT_MONTHS,SECURITY_DEPOSIT_MONTHS_ORIGINAL,APPROVED_DATE,APPLICATION_DATE
count,7051.000000,7051.000000,7051.000000,7051,7051,7051.000000,7051.000000,7051.000000,7051.000000,7051.000000,7051,7051
mean,64112.027230,2016.055879,2046.455396,2026-12-13 08:56:05.650262528,2025-12-31 17:34:12.985392128,11.554673,0.985163,2522.181251,1.245497,1.250035,2025-12-23 06:53:33.501630720,2025-12-20 06:51:55.472982272
min,28202.000000,340.000000,350.000000,2019-01-03 00:00:00,2018-01-08 00:00:00,6.000000,0.964200,340.000000,1.000000,1.000000,2018-01-05 00:00:00,2017-12-31 00:00:00
25%,60607.000000,1310.000000,1335.500000,2026-10-04 00:00:00,2025-12-23 00:00:00,8.000000,0.977800,1430.000000,1.000000,1.000000,2025-12-14 00:00:00,2025-12-10 00:00:00
50%,77036.000000,1700.000000,1735.000000,2027-01-15 00:00:00,2026-03-16 00:00:00,11.000000,0.985300,1970.000000,1.000000,1.000000,2026-03-07 00:00:00,2026-03-04 00:00:00
75%,80209.000000,2290.000000,2331.000000,2027-04-25 12:00:00,2026-04-19 00:00:00,14.000000,0.992700,2925.000000,1.000000,1.500000,2026-04-11 00:00:00,2026-04-08 00:00:00
max,85339.000000,19980.000000,20000.000000,2028-05-27 00:00:00,2026-06-16 00:00:00,24.000000,1.004400,37360.000000,2.000000,2.000000,2026-06-04 00:00:00,2026-05-30 00:00:00
std,20782.356418,1388.040601,1409.229149,NaN,NaN,4.504933,0.008895,2085.053730,0.430412,0.433064,NaN,NaN


# Examining Listingstable Dataset

In [13]:
listingstable.head()

,ID,EVENT,PRICE,EVENT_DATE,LISTING_TYPE,DAYS_ON_MARKET,LISTED_DATE,REMOVED_DATE
0,"915-W-Peachtree-St-NW,-Unit-328,-Atlanta,-GA-3...",Rental Listing,3004,2026-02-22,Standard,1,2026-02-22,NaT
1,"1709-S-5th-St,-Austin,-TX-78704",Rental Listing,2875,2026-02-22,Standard,1,2026-02-22,NaT
2,"2151-Cumberland-Pkwy-SE,-Apt-103,-Atlanta,-GA-...",Rental Listing,1816,2026-02-22,Standard,1,2026-02-22,NaT
3,"95-8th-St-NW,-Apt-507,-Atlanta,-GA-30309",Rental Listing,2015,2026-02-22,Standard,1,2026-02-22,NaT
4,"3707-Roswell-Rd-NE,---5105,-Atlanta,-GA-30342",Rental Listing,1413,2026-02-22,Standard,1,2026-02-22,NaT


In [14]:
listingstable.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118324 entries, 0 to 118323
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   ID              118324 non-null  object        
 1   EVENT           118324 non-null  object        
 2   PRICE           118324 non-null  int64         
 3   EVENT_DATE      118324 non-null  datetime64[ns]
 4   LISTING_TYPE    118324 non-null  object        
 5   DAYS_ON_MARKET  118324 non-null  int64         
 6   LISTED_DATE     118324 non-null  datetime64[ns]
 7   REMOVED_DATE    65502 non-null   datetime64[ns]
dtypes: datetime64[ns](3), int64(2), object(3)
memory usage: 7.2+ MB


In [15]:
listingstable.describe()

,PRICE,EVENT_DATE,DAYS_ON_MARKET,LISTED_DATE,REMOVED_DATE
count,1.183240e+05,118324,118324.000000,118324,65502
mean,2.569783e+05,2025-09-20 23:04:52.931273472,74.175180,2025-09-20 23:04:52.931273472,2025-09-30 06:49:21.912613376
min,3.500000e+02,2017-11-20 00:00:00,1.000000,2017-11-20 00:00:00,2022-06-20 00:00:00
25%,1.650000e+03,2025-07-14 00:00:00,1.000000,2025-07-14 00:00:00,2025-07-10 00:00:00
50%,2.600000e+03,2025-12-05 00:00:00,16.000000,2025-12-05 00:00:00,2025-12-02 00:00:00
75%,3.300000e+05,2026-01-30 00:00:00,84.000000,2026-01-30 00:00:00,2026-01-28 00:00:00
max,3.500000e+07,2026-02-22 00:00:00,3017.000000,2026-02-22 00:00:00,2026-02-22 00:00:00
std,6.716417e+05,NaN,153.021915,NaN,NaN


# Examining Inquirytables Dataset

In [17]:
inquirytable.head()

,CHANNEL,MESSAGE,CONVERTED,INQUIRYID,LEAD_SCORE,LISTING_ID,INQUIRER_NAME,FOLLOW_UP_COUNT,RESPONSE_TIME(HRS),INQUIRY_DATE
0,Zillow,Minute while coach class forget instead dinner...,False,8ad0de54-00d3-459f-a123-96e5bcf42726,9,"4716-N-20th-Ave,-Phoenix,-AZ-85015",Megan Jensen,5,44.2,2026-04-27
1,Zillow,Church enough concern by degree force represen...,False,9dd62610-0c06-4685-9337-608e745b02e3,9,"2918-E-Joan-D-Arc-Ave,-Phoenix,-AZ-85032",John Holt Ii,5,37.1,2026-04-08
2,Zillow,Military marriage firm behavior language proba...,False,b484d452-b4cc-46a1-92ce-bbc6e6fe6865,9,"5350-E-Deer-Valley-Dr,-Unit-1260,-Phoenix,-AZ-...",Cody Williams,5,16.0,2026-02-27
3,Zillow,Group yard listen structure lot agree informat...,False,ca204ad0-0d9c-4668-8e5f-3b8014fd5c81,9,"5513-Makeig-St,-Houston,-TX-77026",Sarah Sherman,5,21.9,2026-01-07
4,Zillow,Term cultural friend exactly north sometimes h...,False,f16e6bf7-0a5d-4152-83c9-8cb5802c36f4,9,"3810-N-Fremont-St,-Apt-3,-Chicago,-IL-60613",Christopher Allen,5,44.5,2026-03-24


In [18]:
inquirytable.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252821 entries, 0 to 252820
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   CHANNEL             252821 non-null  object        
 1   MESSAGE             252821 non-null  object        
 2   CONVERTED           252821 non-null  bool          
 3   INQUIRYID           252821 non-null  object        
 4   LEAD_SCORE          252821 non-null  int64         
 5   LISTING_ID          252821 non-null  object        
 6   INQUIRER_NAME       252821 non-null  object        
 7   FOLLOW_UP_COUNT     252821 non-null  int64         
 8   RESPONSE_TIME(HRS)  252821 non-null  float64       
 9   INQUIRY_DATE        252821 non-null  datetime64[ns]
dtypes: bool(1), datetime64[ns](1), float64(1), int64(2), object(5)
memory usage: 17.6+ MB
